# SDSTrack on VisEvent — Cloud GPU Evaluation

Cloud-agnostic notebook for SDSTrack evaluation on VisEvent.
Works on RunPod, Vast.ai, Lambda, Colab, or any GPU server.

## Key Changes from Colab Version
- **No Google Drive dependency** — everything is local or from Hugging Face
- **No subprocess deadlock** — evaluation runs directly in Python
- **Model stays loaded** — loaded once, reused for all sequences
- **Simple workspace** — configurable via `SDSTRACK_WORKSPACE` env var

## Usage
```bash
# Set workspace (optional, defaults to ./sdstrack)
export SDSTRACK_WORKSPACE=/workspace/sdstrack

# Run evaluation
python sdstrack_eval.py --workspace /workspace/sdstrack
```

## Phase 1: Environment Setup

Clone SDSTrack, apply patches, download models.

In [ ]:
# ============================================================
# P1.1: Setup Environment
# ============================================================
import os
import sys

# Set workspace (change this if needed)
WORKSPACE = os.environ.get('SDSTRACK_WORKSPACE', './sdstrack')
os.environ['SDSTRACK_WORKSPACE'] = WORKSPACE

print(f"Workspace: {WORKSPACE}")
print(f"Python: {sys.version}")

# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} ({torch.version.cuda if torch.cuda.is_available() else 'N/A'})")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# P1.2: Run Setup (clone, patch, download models)
# ============================================================
# Run the setup portion of the script
import sdstrack_eval

paths = sdstrack_eval.Paths(sdstrack_eval.get_workspace())
paths.ensure_dirs()

sdstrack_eval.clone_sdstrack(paths)
sdstrack_eval.apply_patches(paths)
sdstrack_eval.download_models(paths)

print("\n[Setup] Complete!")

## Phase 2: Dataset Preparation

Load the list of test tar shards from Hugging Face.

In [ ]:
# ============================================================
# P2.1: Load Dataset List
# ============================================================
tar_files = sdstrack_eval.load_tar_list(paths)
print(f"Total test tar files: {len(tar_files)}")
print(f"First 5: {tar_files[:5]}")

## Phase 3: Streaming Evaluation

Run the evaluation loop. This processes one tar shard at a time,
evaluates sequences directly (no subprocess), and saves progress.

In [ ]:
# ============================================================
# P3.1: Run Evaluation
# ============================================================
# You can run the full script from the notebook:
!python sdstrack_eval.py --workspace {WORKSPACE}

# Or run the evaluation directly (shown below):

In [ ]:
# ============================================================
# P3.2: Direct Evaluation (Alternative to script)
# ============================================================
# This runs the evaluation directly without subprocess
# Progress is saved after each tar shard

import json
import shutil
from pathlib import Path

# Load or create progress
if paths.progress.exists():
    progress = json.loads(paths.progress.read_text())
else:
    progress = {"completed_tars": [], "completed_seqs": []}

pending_seqs = set()
BATCH_SIZE = 10
total_tars = len(tar_files)

for i, tar_name in enumerate(tar_files):
    if tar_name in progress['completed_tars']:
        continue
    
    print(f"\n{'='*60}")
    print(f"[{i+1}/{total_tars}] {tar_name}")
    
    # Download
    tar_path = sdstrack_eval.hf_hub_download(
        repo_id=sdstrack_eval.HF_DATASET_REPO,
        filename=f"{sdstrack_eval.WEBDATASET_PATH}/{tar_name}",
        repo_type="dataset",
        cache_dir=paths.cache,
    )
    
    # Extract
    new_seqs = sdstrack_eval.extract_tar(Path(tar_path), paths)
    pending_seqs.update(new_seqs)
    
    # Clean cache
    shutil.rmtree(paths.cache, ignore_errors=True)
    paths.cache.mkdir(parents=True, exist_ok=True)
    
    # Evaluate batch
    if len(pending_seqs) >= BATCH_SIZE or (i == total_tars - 1 and pending_seqs):
        batch = sorted(pending_seqs)
        print(f"\n[Eval] {len(batch)} sequences...")
        
        done = sdstrack_eval.run_evaluation_direct(paths, batch)
        
        for seq in done:
            seq_dir = paths.test_subset / seq
            if seq_dir.exists():
                shutil.rmtree(seq_dir, ignore_errors=True)
            pending_seqs.discard(seq)
        
        progress['completed_seqs'].extend(done)
        print(f"[Eval] Done: {len(done)}/{len(batch)}")
    
    # Save progress
    progress['completed_tars'].append(tar_name)
    paths.progress.write_text(json.dumps(progress, indent=2))
    print(f"[Progress] {len(progress['completed_tars'])}/{total_tars} tars")

# Final batch
if pending_seqs:
    batch = sorted(pending_seqs)
    done = sdstrack_eval.run_evaluation_direct(paths, batch)
    progress['completed_seqs'].extend(done)
    paths.progress.write_text(json.dumps(progress, indent=2))

print(f"\nComplete! {len(set(progress['completed_seqs']))} sequences done.")

## Phase 4: Metrics

Compute Success AUC and Precision @ 20px.

In [ ]:
# ============================================================
# P4.1: Compute Metrics
# ============================================================
import os
import numpy as np
import glob

RESULTS_DIR = paths.results
GT_BASE = paths.test_subset

def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi1 = max(x1, x2); yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2); yi2 = min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0

def compute_metrics(results_dir, gt_base):
    if not os.path.exists(results_dir):
        print("ERROR: Results directory not found."); return
    files = sorted(glob.glob(os.path.join(results_dir, "*.txt")))
    if not files:
        print("ERROR: No result files found."); return
    print(f"Processing {len(files)} result files...")
    all_ious, all_dists = [], []
    for res_file in files:
        seq = os.path.basename(res_file).replace(".txt", "")
        gt_file = os.path.join(gt_base, seq, "groundtruth.txt")
        if not os.path.exists(gt_file):
            print(f"  Skipping {seq} (no groundtruth)"); continue
        try:
            pred = np.loadtxt(res_file, delimiter=",")
            gt = np.loadtxt(gt_file, delimiter=",")
        except Exception:
            print(f"  Skipping {seq} (load error)"); continue
        if pred.ndim == 1: pred = pred.reshape(1, -1)
        if gt.ndim == 1: gt = gt.reshape(1, -1)
        n = min(len(pred), len(gt))
        pred, gt = pred[:n], gt[:n]
        for p, g in zip(pred, gt):
            all_ious.append(compute_iou(p, g))
            pcx, pcy = p[0] + p[2]/2, p[1] + p[3]/2
            gcx, gcy = g[0] + g[2]/2, g[1] + g[3]/2
            all_dists.append(np.sqrt((pcx - gcx)**2 + (pcy - gcy)**2))
    
    thresholds = np.arange(0, 1.01, 0.01)
    success = [np.mean(np.array(all_ious) >= t) for t in thresholds]
    auc = np.mean(success)
    prec = np.mean(np.array(all_dists) <= 20)
    
    print(f"\n{'='*60}")
    print(f"Success AUC:  {auc:.4f}")
    print(f"Precision:    {prec:.4f}")
    print(f"{'='*60}")
    return auc, prec

compute_metrics(RESULTS_DIR, GT_BASE)

## Appendix

### Quick commands

```bash
# Run full evaluation from terminal
python sdstrack_eval.py --workspace /workspace/sdstrack

# Check progress
cat /workspace/sdstrack/progress.json

# Count results
ls /workspace/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe/*.txt | wc -l
```

### File locations

| Item | Path |
|------|------|
| Workspace | `$SDSTRACK_WORKSPACE` or `./sdstrack` |
| Data | `{workspace}/data/visevent/test/test_subset` |
| Results | `{workspace}/RGBE_workspace/results/VisEvent/cvpr2024_rgbe/` |
| Progress | `{workspace}/progress.json` |
| Models | `{workspace}/models/` |
| Pretrained | `{workspace}/pretrained/` |